In [45]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import (
    MinMaxScaler,
    OneHotEncoder,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [18]:
def create_spark():
    """ Create a SparkSession object. """
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("TestSuite") \
        .config(key='spark.sql.shuffle.partitions', value='4') \
        .config(key='spark.default.parallelism', value='4') \
        .config(key='spark.sql.session.timeZone', value='UTC') \
        .config(key='spark.ui.enabled', value='false') \
        .config(key='spark.app.id', value='Test') \
        .config(key='spark.driver.host', value='localhost') \
        .getOrCreate()

    return spark

In [19]:
spark = create_spark()

In [20]:
path_to_data = '../../dataset/CarPrice_Assignment.csv'

car_spark_df = spark.read.csv(path_to_data, header=True, inferSchema=True)

In [21]:
features = ['symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight',
            'enginesize', 'boreratio', 'stroke', 'compressionratio', 'horsepower',
            'peakrpm', 'citympg', 'highwaympg']
target = 'price'

In [22]:
car_spark_df.show(5)

+------+---------+--------------------+--------+----------+----------+-----------+----------+--------------+---------+---------+--------+---------+----------+----------+--------------+----------+----------+---------+------+----------------+----------+-------+-------+----------+-------+
|car_ID|symboling|             CarName|fueltype|aspiration|doornumber|    carbody|drivewheel|enginelocation|wheelbase|carlength|carwidth|carheight|curbweight|enginetype|cylindernumber|enginesize|fuelsystem|boreratio|stroke|compressionratio|horsepower|peakrpm|citympg|highwaympg|  price|
+------+---------+--------------------+--------+----------+----------+-----------+----------+--------------+---------+---------+--------+---------+----------+----------+--------------+----------+----------+---------+------+----------------+----------+-------+-------+----------+-------+
|     1|        3|  alfa-romero giulia|     gas|       std|       two|convertible|       rwd|         front|     88.6|    168.8|    64.1|  

In [23]:
raw_string_columns = ['fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem']
indexed_string_columns = [col + "_index" for col in raw_string_columns]
encoded_string_columns = [col + "_ohe" for col in raw_string_columns]

In [24]:
indexer = StringIndexer(
    inputCols=raw_string_columns,
    outputCols=indexed_string_columns,
    handleInvalid="keep"  # optional
)
car_spark_df = indexer.fit(car_spark_df).transform(car_spark_df)

In [26]:
encoder = OneHotEncoder(
    inputCols=indexed_string_columns,
    outputCols=encoded_string_columns
)
car_spark_df = encoder.fit(car_spark_df).transform(car_spark_df)

In [53]:
car_spark_df.select('doornumber', 'doornumber_index', 'doornumber_ohe').show(5)

+----------+----------------+--------------+
|doornumber|doornumber_index|doornumber_ohe|
+----------+----------------+--------------+
|       two|             1.0| (2,[1],[1.0])|
|       two|             1.0| (2,[1],[1.0])|
|       two|             1.0| (2,[1],[1.0])|
|      four|             0.0| (2,[0],[1.0])|
|      four|             0.0| (2,[0],[1.0])|
+----------+----------------+--------------+
only showing top 5 rows


In [28]:
vectorizer = VectorAssembler(inputCols=features+indexed_string_columns, outputCol='features')
car_spark_df = vectorizer.transform(car_spark_df)

In [30]:
scaler = MinMaxScaler(inputCol='features', outputCol='scaled_features').fit(car_spark_df)
scaled_data = scaler.transform(car_spark_df)

In [54]:
scaled_data.select('features', 'scaled_features').show(5)

+--------------------+--------------------+
|            features|     scaled_features|
+--------------------+--------------------+
|[3.0,88.6,168.8,6...|[1.0,0.0583090379...|
|[3.0,88.6,168.8,6...|[1.0,0.0583090379...|
|[1.0,94.5,171.2,6...|[0.60000000000000...|
|[2.0,99.8,176.6,6...|[0.8,0.3848396501...|
|[2.0,99.4,176.6,6...|[0.8,0.3731778425...|
+--------------------+--------------------+
only showing top 5 rows


In [32]:
train, test = scaled_data.randomSplit([0.7, 0.3], seed=42)

In [33]:
dt = DecisionTreeRegressor(featuresCol='scaled_features', labelCol='price')
model = dt.fit(train)

In [34]:
predicted = model.transform(test)

In [48]:
predicted = (predicted.withColumn('prediction', F.round(F.col('prediction'), 2))
                      .withColumn('error', F.round(F.col('price') - F.col('prediction'), 2))
                      .withColumn('error_percentage', F.round(F.col('error') / F.col('price') * 100, 2)))
predicted.select('price', 'prediction', 'error', 'error_percentage').show(5)

+---------+----------+--------+----------------+
|    price|prediction|   error|error_percentage|
+---------+----------+--------+----------------+
|  16500.0|  14812.13| 1687.87|           10.23|
|  17710.0|  19411.67|-1701.67|           -9.61|
|  23875.0|  19411.67| 4463.33|           18.69|
|17859.167|   16278.0| 1581.17|            8.85|
|  21105.0|  14812.13| 6292.87|           29.82|
+---------+----------+--------+----------------+
only showing top 5 rows


In [49]:
avg_percentage_error = predicted.select(F.mean(F.abs(F.col('error_percentage')))).collect()[0][0]
print(f'Average percentage error: {avg_percentage_error:.2f}%')

Average percentage error: 15.31%


In [51]:
avg_error = predicted.select(F.mean(F.abs(F.col('error')))).collect()[0][0]
print(f'Average error: {avg_error:.2f}') # AKA Mean Absolute Error (MAE)

Average error: 2199.61


In [50]:
metrics = {
    "r2": RegressionEvaluator(metricName="r2"),
    "rmse": RegressionEvaluator(metricName="rmse"),
    "mae": RegressionEvaluator(metricName="mae"),
}

for name, evaluator in metrics.items():
    evaluator.setLabelCol(target)
    print(f'Metric: {name} =  {evaluator.evaluate(predicted)}')

# Metric: r2 =  0.8549875063504369
# Metric: rmse =  2173.1015208129197
# Metric: mae =  1616.6682997070607

Metric: r2 =  0.8430067849007103
Metric: rmse =  3197.4088977935367
Metric: mae =  2199.607842105263
